<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/JARVIS_QuantumEspressoColab_Designing_High_Tc_Superconductors_with_BCS_inspired_Screening%2C_Density_Functional_Theory_and_Deep_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[Designing High-Tc Superconductors with BCS-inspired Screening, Density Functional Theory and Deep-learning](https://arxiv.org/abs/2205.00060)

In [ ]:
!pip install jarvis-tools

In [ ]:
import os
# Clone the QE git
if not os.path.exists('q-e'):
   os.system('git clone https://github.com/QEF/q-e.git')
# Install the dependencies
!apt-get install -y libfftw3-3 libfftw3-dev libfftw3-doc
%cd q-e
# run the configure file
!DFLAGS='-D__OPENMP -D__FFTW3 -D__MPI -D__SCALAPACK' FFT_LIBS='-lfftw3'  ./configure --enable-openmp
! make pw

In [ ]:
!make ph

In [ ]:
!lscpu 

In [ ]:
from jarvis.db.figshare import get_jid_data
from jarvis.tasks.qe.super import SuperCond
from jarvis.core.utils import get_factors
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import data, get_jid_data
from jarvis.core.kpoints import Kpoints3D
from jarvis.tasks.queue_jobs import Queue
from jarvis.db.jsonutils import dumpjson
import os
from jarvis.analysis.structure.spacegroup import Spacegroup3D
import glob
from jarvis.db.jsonutils import loadjson

jid = 'JVASP-816' #Aluminum
dat = get_jid_data(jid=jid, dataset="dft_3d")

In [ ]:
def non_prime_kpoints(kpts=[]):
    """Get non prime kpoints."""
    mem = []
    for i in kpts:
        facts = get_factors(i)
        if len(facts) == 1:
            val = i + 1
        else:
            val = i
        mem.append(val)
    return mem

qe_cmd = "/content/q-e/bin/pw.x"
a_atoms = Atoms.from_dict(dat["atoms"])
atoms = Spacegroup3D(a_atoms).refined_atoms.get_primitive_atoms
kp = Kpoints3D().automatic_length_mesh(lattice_mat=atoms.lattice_mat, length=dat["kpoint_length_unit"])
kpts = kp._kpoints[0]
kpts=non_prime_kpoints(kpts)
kp=Kpoints3D(kpoints=[kpts])
print ('kpts',kpts)

nq1 = get_factors(kpts[0])[0]
nq2 = get_factors(kpts[1])[0]
nq3 = get_factors(kpts[2])[0]
qp = Kpoints3D(kpoints=[[nq1, nq2, nq3]])

sup = SuperCond(atoms=atoms, kp=kp, qp=qp,qe_cmd=qe_cmd)

In [ ]:
sup.to_dict()

In [ ]:
sup.runjob()

In [ ]:
!ls

In [ ]:
import os, glob
from jarvis.core.atoms import Atoms
from jarvis.db.jsonutils import loadjson, dumpjson
from jarvis.core.spectrum import Spectrum
from jarvis.io.qe.outputs import DataFileSchema
import numpy as np
import glob

RY_TO_EV = 13.6056980659
def calc_Tc(wlog=300, lamb=1.0, mu=0.1): #mu 0.09
    """Calculate Tc."""

    tc = (wlog / 1.2) * np.exp(
        -1.04 * (1 + lamb) / (lamb - mu * (1 + 0.062 * lamb))
    )
    return tc

def get_a2f(fname="a2F.dos10_JVASP-961"):
    f = open(fname, "r")
    lines = f.read().splitlines()
    f.close()
    x = []
    y = []
    for i in lines:
        tmp = i.split()
        if len(tmp) > 2 and "#" not in i and "lambda" not in i:
            tmp = i.split()
            # print(tmp)
            x.append(1000 * RY_TO_EV * float(tmp[0]))
            y.append(float(tmp[1]))
    return x, y

def parse_lambda(filename="lambda"):
    """Parse lambda file."""
    f = open(filename, "r")
    lines = f.read().splitlines()
    f.close()
    fname = filename.replace("lambda", "QE.save/data-file-schema.xml")
    print("fname", fname)
    ds = DataFileSchema(fname)  
    fname1 = filename.replace("lambda", "a2F.dos10")
    ax, ay = get_a2f(fname1)
    # new_dist = np.arange(0.0, 100, 0.1)
    # new_dist = np.arange(0.0, 100, 0.5)
    new_dist = np.arange(0.0, 100, 1)
    s = Spectrum(x=ax, y=ay)
    interp = s.get_interpolated_values(new_dist=new_dist)

    x = []
    for i in lines:
        if "Broadening" in i and '*****' not in i:
         try:
            tmp = i.split()
            # print(i.split())
            wlog = float(tmp[-1])
            # dos=float(tmp[4])
            lamb = float(tmp[3])
            Tc = calc_Tc(wlog=wlog, lamb=lamb)
            # print("Tc", Tc)
            x.append(Tc)
            # print()
         except Exception as exp:
             print ('Filoename',filename,exp)
             pass
    atoms = ds.final_structure  # .composition.reduced_formula
    # interp=interp/(3*atoms.num_atoms)
    # tc=float(x[-1]))
    return atoms, wlog, lamb, Tc, jid, interp.tolist(),ax,ay
atoms, wlog, lamb, Tc, jid, interp,ax,ay = parse_lambda('./lambda')

In [ ]:
atoms

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(ax,ay)
plt.xlabel('Freq(meV)')
plt.ylabel('a2F')

In [ ]:
print('Tc K:', Tc)